# 3. Model Analysis

This notebook analyzes model predictions and investigates individual creatures.

**Input:**
- `helper_files/engineered_features.parquet`
- `pickled_models/hp_model_cr*.pkl`

**Output:**
- `data/engineered_features.csv`
- `data/feature_contributions.csv`

## Imports and Configs

In [1]:
import numpy as np
import os
import pandas as pd
import sys

from pathlib import Path

In [2]:
# Detect execution context and set paths dynamically
sys.path.insert(0, '.')

# Get the current working directory
cwd = Path.cwd()

# Check if we're in the notebooks directory or project root
if cwd.name == 'notebooks':
    # Running from notebooks directory (in Jupyter)
    DATA_DIR = '../data'
    PICKLED_MODELS_DIR = '../pickled_models'
    MONSTER_BUILDER_DIR = '../monster-builder-v2'
    HELPERS_DIR = './helper_files'
    IO_DIR = './notebooks_io'
    IN_NB_DIR = False
else:
    # Running from project root (via run_three_tier_model.py)
    DATA_DIR = './data'
    PICKLED_MODELS_DIR = './pickled_models'
    MONSTER_BUILDER_DIR = './monster-builder-v2'
    HELPERS_DIR = './notebooks/helper_files'
    IO_DIR = './notebooks/notebooks_io'
    IN_NB_DIR = False

print(f"📁 Execution context detected:")
print(f"   Current directory: {cwd}")
print(f"   Data directory: {DATA_DIR}")
print(f"   Models directory: {PICKLED_MODELS_DIR}")

print("Imports successful")

📁 Execution context detected:
   Current directory: /workspaces/matrix_v0
   Data directory: ./data
   Models directory: ./pickled_models
Imports successful


In [13]:
# Add helper_files to path
if IN_NB_DIR is True:
    from helper_files import (
        load_model, get_phase3_features, investigate_creature, summarize_model_performance
    )

    print("Imports successful")
else:
    from notebooks.helper_files import (
        load_model, get_phase3_features, investigate_creature, summarize_model_performance
    )

    print("Imports successful")


Imports successful


## Load Data and Models

In [4]:
import_path = DATA_DIR + '/engineered_features.csv'
df = pd.read_csv(import_path)

In [5]:
import_path = DATA_DIR + '/feature_contributions.csv'
contributions_df = pd.read_csv(import_path)

In [15]:
phase3_features = get_phase3_features()

In [22]:
results = {}
load_path = PICKLED_MODELS_DIR + "/hp_model_tier.pkl"
# Save each model
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:

    filepath = load_path.replace('tier', tier)
    results[tier] = load_model(filepath)
    print(f"Loaded {tier} model to {filepath}")

print("\nAll models loaded successfully!")

Loaded cr1 model to ./pickled_models/hp_model_cr1.pkl
Loaded cr2 model to ./pickled_models/hp_model_cr2.pkl
Loaded cr3 model to ./pickled_models/hp_model_cr3.pkl
Loaded cr4 model to ./pickled_models/hp_model_cr4.pkl
Loaded cr5 model to ./pickled_models/hp_model_cr5.pkl

All models loaded successfully!


In [23]:
results['cr1'].keys()


dict_keys(['model', 'scaler', 'feature_names', 'train_count', 'test_r2', 'test_mae'])

## Model Analysis

In [24]:
summarize_model_performance(results)


5-BUCKET HP MODEL TRAINING COMPLETE

MODEL PERFORMANCE SUMMARY:

   CR < 1 Model:
      Training samples: 113
      Test R²:  0.4097
      Test MAE: 4.98 HP

   CR 1-4 Model:
      Training samples: 99
      Test R²:  0.5658
      Test MAE: 11.48 HP

   CR 5-10 Model:
      Training samples: 65
      Test R²:  0.4799
      Test MAE: 18.28 HP

   CR 11-16 Model:
      Training samples: 27
      Test R²:  0.9807
      Test MAE: 2.98 HP

   CR > 16 Model:
      Training samples: 20
      Test R²:  0.9533
      Test MAE: 16.04 HP

All 5 models trained successfully!


In [25]:
# Display top feature coefficients for each tier
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:
    print(f"\n{tier.upper()} Top Features by Coefficient:")
    coefs = results[tier]['model'].coef_
    coef_df = pd.DataFrame({
        'feature': phase3_features,
        'coefficient': coefs
    }).sort_values('coefficient', key=abs, ascending=False)
    
    print(coef_df.head(10).to_string(index=False))


CR1 Top Features by Coefficient:
                 feature  coefficient
     speed_fly_deviation     2.434494
condition_immunity_count     2.356248
  speed_ground_deviation     2.280135
     total_ability_count    -1.654340
      inflicts_petrified    -1.512433
      inflicts_paralyzed    -1.119809
     vulnerability_count    -0.944148
       spellcaster_level    -0.886919
  save_proficiency_count     0.787243
             speed_climb    -0.731597

CR2 Top Features by Coefficient:
                    feature  coefficient
          spellcaster_level    -5.427423
     save_proficiency_count     5.154580
    skill_proficiency_count    -3.644197
        vulnerability_count     3.563618
                speed_climb    -3.278089
           inflicts_charmed    -3.092256
                 speed_swim     3.061025
                trait_count     2.596074
has_magic_resistance_scaled    -2.564761
               speed_burrow    -2.461672

CR3 Top Features by Coefficient:
                 feature  coe

## Investigate Specific Creatures

In [6]:
# Interactive investigation
# Uncomment and modify to investigate specific creatures:
# investigate_creature('Adult Red Dragon', df, contributions_df)
# investigate_creature('Tarrasque', df, contributions_df)
investigate_creature('Goblin', df, contributions_df)

  GOBLIN (CR 0.25)

Actual HP:               7
Predicted HP:            9
Error:                   2  (  35.3%)

--------------------------------------------------------------------------------

PHASE 1: CR BASELINE
  HP Baseline (CR 0.25):                                20

PHASE 1.5: RESISTANCES & IMMUNITIES
  No resistances or immunities
  HP after Phase 1.5:                                    20

PHASE 2: COMBAT STATS
  AC Contribution                                     -6
  Attack Bonus Contribution                           -2
  DPR Contribution                                    -0
  -----------------------------------------------------
  Phase 2 Total:                                         -8
  HP after Phase 2:                                      11

PHASE 3: INDIVIDUAL FEATURES
  Feature: has_legendary_resistance_scaled     value:    N/A  hp impact:     -0
  Feature: has_magic_resistance_scaled         value:    N/A  hp impact:     -0
  Feature: has_regeneration_scaled   

In [7]:
# List worst predictions
print("\nWorst Over-predictions (predicted > actual):")
over_pred = df[df['hp_delta'] > 0].nlargest(10, 'hp_delta_pct')[['Name', 'cr_numeric', 'actual_hp', 'predicted_hp', 'hp_delta_pct']]
print(over_pred.to_string(index=False))


Worst Over-predictions (predicted > actual):
           Name  cr_numeric  actual_hp  predicted_hp  hp_delta_pct
           Frog       0.000          1      7.831192    683.119232
            Rat       0.000          1      7.010138    601.013787
         Sprite       0.250          2      9.586363    379.318170
            Owl       0.000          1      4.610816    361.081555
Poisonous Snake       0.125          2      9.053257    352.662868
      Sea Horse       0.000          1      4.483275    348.327485
         Lizard       0.000          2      7.192600    259.629998
           Crab       0.000          2      6.877192    243.859593
            Cat       0.000          2      6.514010    225.700494
         Quasit       1.000          7     18.428259    163.260849


In [8]:
print("\nWorst Under-predictions (predicted < actual):")
under_pred = df[df['hp_delta'] < 0].nsmallest(10, 'hp_delta_pct')[['Name', 'cr_numeric', 'actual_hp', 'predicted_hp', 'hp_delta_pct']]
print(under_pred.to_string(index=False))


Worst Under-predictions (predicted < actual):
      Name  cr_numeric  actual_hp  predicted_hp  hp_delta_pct
   Quipper         0.0          1    -12.879311  -1387.931130
    Weasel         0.0          1     -5.101835   -610.183508
    Spider         0.0          1     -4.547813   -554.781268
   Octopus         0.0          3     -8.436675   -381.222515
      Hawk         0.0          1     -1.615406   -261.540650
     Raven         0.0          1     -0.952220   -195.221990
Homunculus         0.0          5     -3.006450   -160.128990
       Bat         0.0          1     -0.524688   -152.468826
    Baboon         0.0          3     -1.329291   -144.309693
     Eagle         0.0          3     -1.146064   -138.202142


In [9]:
print("\nBest Predictions:")
df_check_best = df[['Name', 'cr_numeric', 'actual_hp', 'predicted_hp', 'hp_delta_pct']].copy()
df_check_best['hp_delta_pct_abs'] = df_check_best['hp_delta_pct'].abs()
best = df_check_best.nsmallest(10, 'hp_delta_pct_abs')
print(best.to_string(index=False))


Best Predictions:
         Name  cr_numeric  actual_hp  predicted_hp  hp_delta_pct  hp_delta_pct_abs
    Pit Fiend      20.000        300    300.000002  5.060586e-07      5.060586e-07
    Ice Devil      14.000        180    180.000047  2.635028e-05      2.635028e-05
      Erinyes      12.000        153    152.999928 -4.730817e-05      4.730817e-05
      Vampire      13.000        144    143.999931 -4.774074e-05      4.774074e-05
        Balor      19.000        262    262.000145  5.520084e-05      5.520084e-05
     Basilisk       3.000         52     52.000029  5.554708e-05      5.554708e-05
Dragon Turtle      17.000        341    341.000213  6.235316e-05      6.235316e-05
       Kraken      23.000        472    472.000380  8.058110e-05      8.058110e-05
   Cockatrice       0.500         27     27.000028  1.046305e-04      1.046305e-04
      Cultist       0.125          9      8.999988 -1.377728e-04      1.377728e-04


# Deatiled Model Breakdown